# AT&T Tienda Online — Scraper de celulares

Sitio: https://att.com.mx/market/destacados

El sitio es una app Next.js que carga el catálogo desde la API de Infonova (`commerceSearch/v1/resources`).
En lugar de abrir un navegador, este notebook llama directamente a esa API (un `POST` con cuerpo `{}`),
así que es rápido y no depende del HTML.

Cada producto trae sus **variantes** (combinación de capacidad + color), cada una con su propio SKU y precios:

- `LIST_PRICE` → precio original (solo existe cuando hay descuento)
- `PRICE` / `SELLING_PRICE` → precio de venta
- `DISCOUNT` → % de descuento publicado

La salida usa el mismo formato que el scraper de AT&T (`Daily Pricing Mapping ATT`).

In [1]:
import re
import time
from datetime import datetime

import pandas as pd
import requests

try:
    from zoneinfo import ZoneInfo
    TZ_CDMX = ZoneInfo("America/Mexico_City")
except Exception:
    TZ_CDMX = None

# ===================== CONFIGURACIÓN =====================
API_URL = "https://us02.infonova.com/r6-api/PROD_ATT_MX/commerceSearch/v1/resources"
SITE_URL = "https://att.com.mx"
STORE_URL = SITE_URL + "/market/destacados"
PAGE_SIZE = 100

# Tipos de producto a incluir y su categoría en la salida
TEMPLATES = {"MOBILE_PHONE": "Smartphone", "MOBILE_ACCESSORIES": "Accessories"}

# La API no distingue tipos de accesorio: se clasifican por palabras clave en el nombre
CLAVES_CATEGORIA = {
    "Headphone": ["AUDÍFONOS", "AUDIFONOS", "BUDS", "FREECLIP", "EARBUDS", "AIRPODS"],
    "Smartwatch": ["WATCH", "SMARTBAND"],
}

HEADERS = {
    "Content-Type": "application/json; charset=utf-8",
    "Accept": "application/json",
    "Origin": SITE_URL,
    "Referer": STORE_URL + "/",
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/140.0 Safari/537.36"),
}

SCHEMA = [
    "Marca", "Modelo", "Nombre", "Color", "Capacidad", "SKU", "URL",
    "FechaConsulta", "Precio_original", "Precio_descuento", "Porcentaje_descuento", "Categoria",
]

# La red corporativa intercepta SSL (certificado autofirmado). Si está instalado
# `truststore` se usan los certificados de Windows; si no, se desactiva la verificación.
try:
    import truststore
    truststore.inject_into_ssl()
    VERIFY_SSL = True
except ImportError:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    VERIFY_SSL = False

In [2]:
# ===================== FUNCIONES AUXILIARES =====================
def limpiar_precio(valor) -> str:
    """'$7,499.01' -> '7499.01'. Devuelve '' si no hay número."""
    if valor is None or valor == "":
        return ""
    if isinstance(valor, (int, float)):
        return f"{float(valor):.2f}"
    txt = str(valor).replace("MXN", "").replace("$", "").strip()
    txt = re.sub(r"[^\d,.\-]", "", txt).replace(",", "")
    m = re.search(r"\d+(?:\.\d+)?", txt)
    return m.group(0) if m else ""


def _a_float(precio):
    try:
        return float(precio) if precio not in ("", "No discount", "No inventory", None) else None
    except (TypeError, ValueError):
        return None


def resolver_precios(precio_lista, precio_oferta, descuento_publicado=""):
    """Misma lógica de negocio que el resto de los scrapers."""
    p_ori = limpiar_precio(precio_lista)
    p_desc = limpiar_precio(precio_oferta)

    if not p_ori and p_desc:
        p_ori, p_desc = p_desc, "No discount"
    elif p_ori and not p_desc:
        p_desc = "No discount"
    elif not p_ori and not p_desc:
        p_ori, p_desc = "No inventory", "No discount"

    f_ori, f_desc = _a_float(p_ori), _a_float(p_desc)
    if f_ori is not None and f_desc is not None and f_desc >= f_ori:
        p_desc, f_desc = "No discount", None

    porcentaje = "No discount"
    pub = limpiar_precio(str(descuento_publicado).replace("-", ""))
    if f_ori and f_desc and f_ori > f_desc:
        porcentaje = f"{round(100 * (f_ori - f_desc) / f_ori, 2)}%"
    elif pub and f_desc:
        try:
            v = float(pub)
            if 0 < v <= 100:
                porcentaje = f"{round(v, 2)}%"
        except ValueError:
            pass
    return p_ori, p_desc, porcentaje


def normalizar_capacidad(cap: str) -> str:
    """'256GB' / '256 gb' -> '256 GB'; '1TB' -> '1 TB'."""
    if not cap:
        return ""
    m = re.search(r"(\d+)\s*(GB|TB|MB)", str(cap), re.IGNORECASE)
    return f"{m.group(1)} {m.group(2).upper()}" if m else str(cap).strip()


def construir_modelo(nombre: str, marca: str) -> str:
    """Quita la marca del inicio del nombre: 'Samsung Galaxy A16' -> 'Galaxy A16'."""
    modelo = (nombre or "").strip()
    if marca and modelo.upper().startswith(marca.upper()):
        modelo = modelo[len(marca):].strip(" -")
    return modelo or nombre


def clasificar(prod: dict) -> str:
    """Smartphone / Headphone / Smartwatch / Accessories."""
    categoria = TEMPLATES[prod["template"]]
    if categoria == "Accessories":
        texto = f"{prod.get('name', '')} {prod.get('displayName', '')}".upper()
        for cat, claves in CLAVES_CATEGORIA.items():
            if any(c in texto for c in claves):
                return cat
    return categoria


def obtener_catalogo() -> list:
    """Descarga todas las páginas del catálogo desde la API."""
    productos, page = [], 1
    with requests.Session() as s:
        while True:
            for intento in range(3):
                try:
                    r = s.post(
                        API_URL,
                        params={"page": page, "pageSize": PAGE_SIZE,
                                "sortBy": "parameters.RANK", "sortOrder": "asc"},
                        json={}, headers=HEADERS, timeout=30, verify=VERIFY_SSL,
                    )
                    r.raise_for_status()
                    data = r.json()
                    break
                except Exception as e:
                    print(f"Página {page}, intento {intento + 1} falló: {e}")
                    time.sleep(3)
            else:
                raise RuntimeError(f"No se pudo descargar la página {page}")

            lote = data.get("list", [])
            productos.extend(lote)
            total = data.get("paginationInformation", {}).get("totalNumberOfRecords")
            print(f"Página {page}: {len(lote)} productos (total en catálogo: {total})")
            if not lote or "next" not in data.get("_links", {}):
                break
            page += 1
    return productos


def aplanar_variantes(productos: list, fecha: str) -> list:
    """Una fila por variante (capacidad + color) con el esquema estándar."""
    filas = []
    for prod in productos:
        if prod.get("template") not in TEMPLATES:
            continue
        pp = prod.get("parameters", {})
        marca = (pp.get("BRAND") or "").strip().upper()
        categoria = clasificar(prod)
        for var in prod.get("variants", []):
            vp = var.get("parameters", {})
            nombre = (var.get("displayName") or prod.get("displayName") or "").strip()
            # Sin LIST_PRICE el sitio muestra solo el precio de venta (sin descuento)
            precio_venta = vp.get("PRICE") or vp.get("SELLING_PRICE")
            p_ori, p_desc, pct = resolver_precios(vp.get("LIST_PRICE"), precio_venta, vp.get("DISCOUNT", ""))
            filas.append({
                "Marca": marca,
                "Modelo": construir_modelo(nombre, marca),
                "Nombre": nombre,
                "Color": (vp.get("COLOR") or "").strip(),
                "Capacidad": normalizar_capacidad(vp.get("STORAGE", "")),
                "SKU": vp.get("SKU", ""),
                "URL": f"{STORE_URL}/products/{prod.get('name', '')}?variant={var.get('name', '')}",
                "FechaConsulta": fecha,
                "Precio_original": p_ori,
                "Precio_descuento": p_desc,
                "Porcentaje_descuento": pct,
                "Categoria": categoria,
            })
    return filas

## Ejecutar scraping

In [3]:
ahora = datetime.now(TZ_CDMX) if TZ_CDMX is not None else datetime.now()
fecha_consulta = ahora.strftime("%Y-%m-%d %H:%M:%S")

productos = obtener_catalogo()
data = aplanar_variantes(productos, fecha_consulta)

df = pd.DataFrame(data, columns=SCHEMA)
print(f"\nVariantes antes de limpiar: {len(df)}")
df = df[df["Nombre"].astype(str).str.len() > 1]
df = df.drop_duplicates(subset=["SKU"], keep="first")
# Accesorios sin capacidad -> "NA"
df.loc[(df["Categoria"] != "Smartphone") & (df["Capacidad"] == ""), "Capacidad"] = "NA"
df = df.sort_values(["Categoria", "Marca", "Modelo", "Capacidad", "Color"]).reset_index(drop=True)
print(f"Variantes después de limpiar: {len(df)}")

Página 1: 100 productos (total en catálogo: 103)
Página 2: 3 productos (total en catálogo: 103)

Variantes antes de limpiar: 178
Variantes después de limpiar: 178


## Exportar

In [4]:
fecha_hoy, hora_hoy = ahora.strftime("%Y-%m-%d"), ahora.strftime("%H%M%S")
xlsx_out = f"Daily Pricing Mapping ATT Tienda Online ({fecha_hoy}) {hora_hoy}.xlsx"

df.to_excel(xlsx_out, index=False)
print(f"Datos exportados a {xlsx_out}")

print("\n=== ESTADÍSTICAS ===")
print(f"Total de variantes: {len(df)}  |  Modelos: {df['Modelo'].nunique()}")
print("\nVariantes por categoría:")
print(df["Categoria"].value_counts())
print("\nVariantes por marca:")
print(df["Marca"].value_counts())
print(f"\nCon descuento: {(df['Porcentaje_descuento'] != 'No discount').sum()}")
display(df.head(15))

Datos exportados a Daily Pricing Mapping ATT Tienda Online (2026-10-20) 102858.xlsx

=== ESTADÍSTICAS ===
Total de variantes: 178  |  Modelos: 97

Variantes por categoría:
Categoria
Smartphone     150
Accessories     21
Headphone        5
Smartwatch       2
Name: count, dtype: int64

Variantes por marca:
Marca
SAMSUNG     42
APPLE       22
HONOR       22
MOTOROLA    20
XIAOMI      19
QUIKCELL    16
OPPO        14
GOOGLE       7
SPECK        6
VIVO         5
ZTE          3
HUAWEI       2
Name: count, dtype: int64

Con descuento: 49


,Marca,Modelo,Nombre,Color,Capacidad,SKU,URL,FechaConsulta,Precio_original,Precio_descuento,Porcentaje_descuento,Categoria
0,APPLE,Adaptador de corriente USB-C de 20 W,Adaptador de corriente USB-C de 20 W,Blanco,NA,N.00722010,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,749.00,No discount,No discount,Accessories
1,QUIKCELL,Cable USB-C a Lightning 1.80 m,Cable USB-C a Lightning 1.80 m,Negro,NA,N.00729150,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,699.00,No discount,No discount,Accessories
2,QUIKCELL,Cargador de pared 25W USB-C,Cargador de pared 25W USB-C,Blanco,NA,N.00732660,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,699.00,No discount,No discount,Accessories
3,QUIKCELL,Cargador de pared doble 24W USB-A,Cargador de pared doble 24W USB-A,Negro,NA,N.00735500,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,589.00,No discount,No discount,Accessories
4,QUIKCELL,Cargador de pared doble 35W USB-A + USB-C,Cargador de pared doble 35W USB-A + USB-C,Blanco,NA,N.00732690,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,799.00,No discount,No discount,Accessories
5,QUIKCELL,Cargador para automóvil doble 37W USB-A + USB-C,Cargador para automóvil doble 37W USB-A + USB-C,Negro,NA,N.00730320,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,699.00,No discount,No discount,Accessories
6,QUIKCELL,Estación de carga PowerFlip 15W,Estación de carga PowerFlip 15W,Negro,NA,N.QP15W2IN1CN,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,849.00,No discount,No discount,Accessories
7,QUIKCELL,Mica líquida transparente para smartphone,Mica líquida transparente para smartphone,Transparente,NA,N.00729720,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,849.00,No discount,No discount,Accessories
8,QUIKCELL,Mica protectora para iPhone 15,Mica protectora para iPhone 15,Transparente,NA,N.00729900,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,749.00,No discount,No discount,Accessories
9,QUIKCELL,Mica protectora para iPhone 15 Plus,Mica protectora para iPhone 15 Plus,Transparente,NA,N.00729930,https://att.com.mx/market/destacados/products/...,2026-10-20 10:28:58,749.00,No discount,No discount,Accessories
